# Findings 

- It's called "invariance" because of what it encourages: z1 and z2 are projections from two different augmented views of the same image. By minimizing the MSE between them, you're forcing the model to produce the same representation regardless of augmentation — that's invariance.
So if you crop, flip, and color-jitter the same image two different ways, the model should still output similar embeddings. The MSE penalizes any difference between the two, pushing them to be identical — invariant to the augmentations.


In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 

from losses import HingeStdLoss,CovarianceLoss,VICRegLoss

### HingeStdLoss

In [2]:
# class HingeStdLoss(nn.Module): 
#     def __init__(
#         self,
#         std_margin: float = 1.0
#     ): 
#         """
#         Encourages each feature to maintain at least a minimum standard devaition 
#         Features with std below the margin incur a penalty of (std_margin - std).
#         Args : 
#             std_margin(float,default=1.0):
#                 Minimum desired standard deviation per feature
#         """
#         super().__init__()
#         self.std_margin = std_margin

#     def forward(self,x : torch.Tensor):
#         """
#         Args : 
#             x : [N,D] where N is number of samples, D is feature dimension
#         Returns : 
#             std_loss : Scalar tensor with the hinge loss on standard devaiations 
#         """
#         x = x - x.mean(dim = 0, keepdim=True)
#         std = torch.sqrt(x.var(dim=0) + 0.0001)
#         std_loss = torch.mean(F.relu(self.std_margin - std))
#         return std_loss 

In [3]:
torch.manual_seed(123)
x = torch.randn(2,4)
print(x)

tensor([[-0.1115,  0.1204, -0.3696, -0.2404],
        [-1.1969,  0.2093, -0.9724, -0.7550]])


In [4]:
var_loss = HingeStdLoss(std_margin=1.0)
var_loss(x)

tensor(0.5946)

### CovarianceLoss 

In [5]:
# class CovarianceLoss(nn.Module): 
#     def __init__(self):
#         """
#         Penalize off-diagonal elements of the covariance matrix to encourage 
#         feature decorrelation. 

#         Normalizes by D * (D - 1) where D is feature Dimensionality 
#         """
#         super().__init__()
    
#     def off_diagonal(self,x):
#         n,m = x.shape 
#         assert n == m 
#         return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
    
#     def forward(self, x: torch.Tensor): 
#         """
#         Args : 
#             x: [N,D] where N is number of samples, D is feature dimension
#         """
#         batch_size = x.shape[0]
#         num_features = x.shape[-1]
#         x = x - x.mean(dim=0,keepdim=True)
#         cov = (x.T @ x) / (batch_size - 1) # [D,D]
#         # calculate off-diagonal loss 
#         cov_loss = self.off_diagonal(cov).pow(2).mean()

#         return cov_loss 
        

In [6]:
cov_loss = CovarianceLoss()
cov_loss(x)

tensor(0.0354)

In [7]:
# class VICRegLoss(nn.Module): 
#     """VICReg loss combining invariance, variance (std), and covariance terms."""

#     def __init__(self,std_coef =1.0, cov_coeff=1.0):
#         super().__init__()
#         self.std_coeff = std_coef
#         self.cov_coeff = cov_coeff
#         self.std_loss_fn = HingeStdLoss(std_margin=1.0)
#         self.cov_loss_fn = CovarianceLoss()

#     def forward(self,z1,z2): 
#         """Compute VICReg loss.

#         Args: 
#             z1 : [B, D] - First projection tensor
#             z2 : [B, D] - Second projction tensor 
        
#         Returns: 
#             dict with keys: loss, invariance_loss, var_loss, cov_loss
#         """
#         # Invariance loss
#         sim_loss =  F.mse_loss(z1,z2)

#         # Variance loss (applied to both views and summed)
#         var_loss = self.std_loss_fn(z1) + self.std_loss_fn(z2)

#         # covariance loss (applied to both views and summed)
#         cov_loss = self.cov_loss_fn(z1) + self.cov_loss_fn(z2)

#         total_loss = sim_loss + (self.std_coeff * var_loss) + (self.cov_coeff * cov_loss)

#         return {
#             "loss" : total_loss,
#             "invariance_loss" : sim_loss,
#             "var_loss" : var_loss,
#             "cov_loss" : cov_loss
#         }

In [8]:
loss_fn = VICRegLoss(std_coeff= 1.0, cov_coeff=80.0)
loss_fn(x,x)

{'loss': tensor(6.8595),
 'invariance_loss': tensor(0.),
 'var_loss': tensor(1.1892),
 'cov_loss': tensor(0.0709)}